# Data Checks and Joins of Health Insurance Data (KK-Daten)

Abbreviations / Abkürzungen: 
* EBM: Einheitlicher Bewertungsmaßstab
* AMB: ambulant
* KH: Krankenhaus
* Stat: Stationär
* OPS: Operationen und Prozeduren
* AU: Arbeitsunfähigkeit
* KG: Krankengeld
* ATC: Anatomisch-Therapeutisch-Chemische Klassifikation
* DDD: Defined Daily Dose

In [5]:
def find_project_root(marker=".git"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Marker '{marker}' not found in any parent directories.")


In [6]:
import pandas as pd


import sys
sys.path.append("..")
sys.path.append("../prediction_src")
from fit_prediction_model import *
from prediction_pipeline import *
from evaluate_prediction_model import *
from params import *

# import sys, os, Path
import os
from pathlib import Path
import re
import difflib
import numpy as np

root_dir = find_project_root()
print(root_dir)

# get project directory and add code to system path
project_path = os.getcwd()
if "\\" in project_path:
    project_path = "/".join(os.getcwd().split("\\")[:-2])
sys.path.append(project_path + "/code")
export_name = "KK_Daten"
path_to_data = f"{root_dir}/data/0_raw/{export_name}"

/mnt/c/Users/Juliana/dev/memaef


In [4]:
# this is where we keep the ICD-10 codes requested from Krankenkassen
code_list_file_path = "data/3_prediction"
code_list_file_name = "2024-12-02_praediktionsmodell_icd10diagnosen_detaillevel_KK.xlsx"

In [7]:
from join_kk_data import *

In [8]:
find_project_root()

PosixPath('/mnt/c/Users/Juliana/dev/memaef')

NameError: name 'code' is not defined

## Load our code lists

Load all codes (ICD, procedures, ATC, ...) from our list (excel file): 2024-12-02_praediktionsmodell_icd10diagnosen_detaillevel_KK.xlsx

In [8]:
icd_list = read_code_list(code_list_file_path, code_list_file_name, 0, "ICD-10-Code")

In [9]:
len(icd_list) # 186

186

In [10]:
ops_list = read_code_list(code_list_file_path, code_list_file_name, 1, "OPS-Code")

In [11]:
len(ops_list) # 33

33

In [12]:
ebm_list = read_code_list(code_list_file_path, code_list_file_name, 2, "EBM-Code")

In [13]:
len(ebm_list) # 14

14

In [14]:
atc_list = read_code_list(code_list_file_path, code_list_file_name, 3, "ATC")

In [15]:
all_codes_list = set(icd_list).union(set(ebm_list)).union(set(ops_list)).union(set(atc_list))

all_codes_list = [str(f) for f in all_codes_list]

In [16]:
all_codes_list.append(np.nan)

In [17]:
len(all_codes_list)

333

In [18]:
pd.to_pickle(all_codes_list, "all_codes_list.pkl")

all_codes_list contains 332 unique codes that we are interested in (including all N80)

## Load health insurance datasets

In [19]:
data_files = os.listdir(path_to_data)
dict_data = {}
for f in data_files:
    df_data = pd.read_csv(path_to_data + "/" + f, sep=";", index_col=0)
    dict_data[f.split(".")[0]] = df_data
    print("- {} ({} VersID - {} Fälle): {}".format(f, len(df_data["VersID"].unique()), len(df_data) ,", ".join(df_data.columns)))

- amb_diagnosen.csv (64570 VersID - 383101 Fälle): VersID, FallID, Diag, Diag_Sicherheit
- amb_ebm.csv (43630 VersID - 97605 Fälle): VersID, FallID, Facharztgruppe, EBM, EBM_Datum
- amb_falldaten.csv (99590 VersID - 2709240 Fälle): VersID, FallID, Beginn, Ende
- amb_kh_diagnosen.csv (1743 VersID - 3133 Fälle): VersID, FallID, Diag, Diag_Sicherheit
- amb_kh_falldaten.csv (13726 VersID - 41141 Fälle): VersID, FallID, Beginn, Ende
- amb_kh_ops.csv (350 VersID - 417 Fälle): VersID, FallID, OPS_Datum, OPS
- arzneimittel.csv (54266 VersID - 235347 Fälle): VersID, FallID, PZN, Anzahl_Faktor, ATC, Verordnung, Abgabe, DDD
- au_kg_diagnosen.csv (20338 VersID - 44475 Fälle): VersID, FallID, Diag
- au_kg_falldaten.csv (47117 VersID - 256634 Fälle): VersID, FallID, AU_Beginn, AU_Ende
- reha.csv (77 VersID - 81 Fälle): VersID, FallID, Beginn, Ende, Diag
- stammdaten.csv (100000 VersID - 100000 Fälle): VersID, Geburtsjahr, Sex
- stat_diagnosen.csv (13778 VersID - 82305 Fälle): VersID, FallID, Diag_Pr

## Join full dataset for checks

Right now, all data are stored in separate tables (ambulant, stationär, ...), plus the codes and dates are stored separately.  

We merge the codes with their according dates (Falldaten) and then combine all tables into one.  

For each merge, the number of unique IDs are printed.

In [20]:
# ambulante Diagnosen
amb_data = merge_dates_and_codes(dict_data, "amb_falldaten","amb_diagnosen", "amb", "Diag")

amb_falldaten:99590, amb_diagnosen:64570
99590


In [21]:
# ambulante EBM Codes
amb_ebm_data = merge_dates_and_codes(dict_data, "amb_falldaten","amb_ebm", "ebm", "EBM")

amb_falldaten:99590, amb_ebm:43630
99590


In [22]:
# ambulante Krankenhaus-Diagnosen
amb_kh_diag_data = merge_dates_and_codes(dict_data, "amb_kh_falldaten","amb_kh_diagnosen", "amb_kh_diag", "Diag")

amb_kh_falldaten:13726, amb_kh_diagnosen:1743
13726


In [23]:
amb_kh_diag_data[~amb_kh_diag_data["code"].isna()]["VersID"].nunique()

1743

In [24]:
# ambulante Krankenhaus-Operationen/Prozeduren
amb_kh_ops_data = merge_dates_and_codes(dict_data, "amb_kh_falldaten","amb_kh_ops", "amb_kh_ops", "OPS")

amb_kh_falldaten:13726, amb_kh_ops:350
13726


In [25]:
# stationäre Operationen/Prozeduren
stat_ops_data = merge_dates_and_codes(dict_data, "stat_falldaten","stat_ops", "stat_ops", "OPS")

stat_falldaten:27367, stat_ops:560
27367


In [26]:
# stationäre Diagnosen - combine primary and secondary into one
stat_data = pd.merge(dict_data["stat_falldaten"], dict_data["stat_diagnosen"],
                    on = ["VersID","FallID"], how="inner")

stat_data_long = pd.melt(stat_data,id_vars=["VersID","FallID","Beginn","Ende"],
       value_vars=["Diag_Prim","Diag_Sek"],
       )

stat_data_long.rename(columns={"variable":"source",
                              "value":"code"},
                              inplace=True)

stat_data_long["source"]=stat_data_long["source"].replace({"Diag_Prim":"stat_prim",
                                 "Diag_Sek":"stat_sek"})

In [28]:
full_data = pd.concat([amb_data,
                       amb_ebm_data,
                      amb_kh_diag_data,
                       amb_kh_ops_data,
                       stat_ops_data,
                      stat_data_long,
                      ])

full_data.shape

(5791033, 13)

In [29]:
atc_data = dict_data["arzneimittel"].copy()
atc_data.rename(columns={"ATC":"code","Verordnung":"Beginn","Abgabe":"Ende"},inplace=True)

In [30]:
full_data = pd.concat([full_data,
                       atc_data,
                      ])


In [32]:
full_data["code"] = fast_format(full_data["code"]) # some formatting so that all strings follow same pattern

In [33]:
full_data["code"] = full_data["code"].str.replace("Z370!", "Z37.0")

In [34]:
full_data = full_data[full_data["code"]!="nan"].copy()

In [35]:
full_df = pd.merge(dict_data["stammdaten"], full_data, how="left")

In [36]:
pd.to_pickle(full_df,"../data/processed/full_df.pkl")

In [37]:
full_df = pd.read_pickle("../data/processed/full_df.pkl")

In [38]:
full_df["VersID"].nunique()

100000

## Filter data

- nonfemale_ids
- birth year between 1990 – 2008
- "G" (gesichert) status endometriosis diagnoses only after 12/2021
- adenomyosis neither counted as predictor nor as endometriosis
- predictors only before 01/2022

In [39]:
nonfemale_ids = dict_data["stammdaten"][lambda x:x["Sex"]==0]["VersID"].tolist()

In [40]:
len(nonfemale_ids)

11

In [41]:
full_df = full_df[~full_df["VersID"].isin(nonfemale_ids)].copy()
# for now, we drop these nonfemale IDs

In [43]:
full_df["Geburtsjahr"].min(),full_df["Geburtsjahr"].max() # all good

(1990, 2008)

In [44]:
full_df["VersID"].nunique()

99989

### Only keep "G" (gesichert) status endometriosis diagnoses after 12/2021

In [45]:
full_df[full_df["code"].astype(str).str.contains("N80")].groupby("source")["Diag_Sicherheit"].value_counts(dropna=False)

source       Diag_Sicherheit
amb          G                  3086
             V                  2355
             A                   242
             Z                   146
amb_kh_diag  NaN                 117
             G                   100
             V                    54
             A                     1
stat_prim    NaN                1518
Name: count, dtype: int64

In [46]:
full_df.replace("N80.0", "adenomyose",inplace=True) # we will not account for adenomyosis

In [49]:
result = (
    full_df[(full_df["Beginn"] >= "2022-01") &
            (full_df["code"].astype(str).str.contains("N80")) &
            (full_df["source"].isin(["amb", "amb_kh_diag"]))]
    .groupby("VersID")
    .apply(lambda x: not x["Diag_Sicherheit"].isin(["G", "Z"]).any())
)

n80_ids_not_g = result[result].index # N80 past 01/2022 that do not have status "G" - those are not counted as cases

/tmp/ipykernel_6935/3500832417.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: not x["Diag_Sicherheit"].isin(["G", "Z"]).any())


In [50]:
len(n80_ids_not_g)

928

In [51]:
n80_ids_before_followup = full_df[
    (full_df["code"].astype(str).str.contains("N80"))
    & (full_df["Beginn"] < "2022-01-01")
&  (~full_df["Diag_Sicherheit"].isin(["V", "A"]))
]["VersID"].unique()
len(n80_ids_before_followup) # N80 vor 01/2022 die gesichert sind - werden ausgeschlossen

119

In [52]:
full_df.replace("adenomyose","N80.0", inplace=True) 

### Drop n80_ids_before_followup 

In [68]:
n80_df = full_df[~full_df["VersID"].isin(n80_ids_before_followup)].copy()

In [69]:
n80_df.loc[(n80_df["code"].astype(str).str.contains("N80")) &
            (n80_df["Beginn"] < "2022-01"), "code"] = np.nan # set all N80 diagnosis during baseline period to NA so that we do not accidentally consider them

In [70]:
n80_df["VersID"].nunique() # 99870 correct

99870

### Set all predictors with status "A" (ausgeschlossen) to NA

In [71]:
n80_df.loc[(n80_df["Beginn"]<"2022-01") & (n80_df["code"].isin(all_codes_list)) & (n80_df["Diag_Sicherheit"]=="A"),"code"] = np.nan

### Set all predictors past baseline period to NA

In [72]:
pred_list = [l for l in all_codes_list if "N80" not in str(l)]

In [73]:
n80_df.loc[(n80_df["Beginn"]>="2022-01") & (n80_df["code"].isin(pred_list)),"code"].shape

(524017,)

In [74]:
n80_df.loc[(n80_df["Beginn"]>="2022-01") & (n80_df["code"].isin(pred_list)),"code"] = np.nan

# Final steps & save

In [76]:
n80_df.replace("adenomyose","N80.0",inplace=True)

In [77]:
n80_df.fillna("999",inplace=True)

/tmp/ipykernel_6935/1695495756.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '999' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  n80_df.fillna("999",inplace=True)


In [78]:
pd.to_pickle(n80_df, "../data/processed/n80_df.pkl")

# Only Menstruationsschmerz

In [3]:
n80_df = pd.read_pickle("../data/processed/n80_df.pkl")

In [79]:
is_sum = False

In [82]:
n80_df = create_target_variables(n80_df)

In [85]:
n80_df['category'] = n80_df['code'].map(CODE_TO_LABEL)

kk_data, group_cols, model_cols = create_binary_predictors(n80_df, GROUP_COLS, GROUP_MODEL_COLS, as_sum = is_sum)

In [87]:
kk_data["Menstruationsschmerzen"].value_counts()

Menstruationsschmerzen
False    87607
True     12263
Name: count, dtype: int64

In [88]:
kk_data[kk_data["Menstruationsschmerzen"]==1].shape

(12263, 12)

In [89]:
kk_data["endo"].value_counts()

endo
False    97781
True      2089
Name: count, dtype: int64

In [1]:
97781+2089

99870

In [90]:
pd.to_pickle(kk_data[kk_data["Menstruationsschmerzen"]==1],"../data/processed/menstruationsschmerz_data.pkl")

# Decisions

- n80: 
  - ambulante Diagnosen nach baseline, die nicht "G" (=gesichert) sind: treat as control
  - Diagnosis N80 während baseline die nicht nur Verdachtsdiagnose oder Ausschlussdiagnose ist: ID komplett entfernen
- IDs, die **nur** Prädiktoren **nach** baseline haben (und keine gültige N80 Diagnose): treat as control
- "Diagnoseart" wird ignoriert
- primäre and sekundäre Diagnose werden nicht unterschieden
- 11 nonfemale IDs werden ausgeschlossen
- Datumsspalten: Verschreibungsdatum for ATC, Beginn for Diagnosen